# Continuation Laboratory (Python / Jupyter port)

A Python reimplementation of the *Rust Continuation Laboratory* — twenty-four small, runnable
experiments from a process-relational research program in which distinctions are maintained,
states are moments in trajectories, admissibility is future-sensitive, and repair preserves
structured possibility.

Python is not offered as evidence that the framework is true, any more than the Rust originals
were. It's a different experimental medium: Rust's ownership and lifetimes make certain
continuation claims *inspectable by the compiler*; Python has no such enforcement, so several
experiments below narrate the constraint explicitly instead of having it checked for you. Where
that difference matters, the experiment says so.

**Method.** Run each experiment's cell (after running the *Shared helpers* cell once). Each one
prints a claim, a computational trace, and a result. The most productive use of the notebook is
to change the parameters until the printed conclusion fails, then decide whether the failure
belongs to the theory, the implementation, or the representation you chose.

**Suggested sequence.** 01, 02, 07, 09, 10, 11, 13, 14, 15, 16, 18, 21 form the core progression
from distinction through admissibility and repair to persistent history and future-sensitive
viability. 22–24 extend it outward: composition of repairs (22), non-local admissibility (23),
and observer-relative restorability (24).


## Shared helpers

Run this cell first — every experiment below depends on it.

In [12]:
def banner(name, claim):
    print(f"EXPERIMENT: {name}")
    print(f"CLAIM: {claim}\n")

def section(name):
    print(f"\n{name}")
    print("-" * len(name))

def bool_word(value):
    return "yes" if value else "no"

## Experiment 01: Distinction Machine

**Claim:** Equal present outputs can preserve different future possibilities.

**Model:** A bare boolean, a resolved-but-unlabeled distinction, and a distinction that keeps its
own history are compared. All three can agree on their current visible output while supporting
different futures.


In [13]:
def exp01():
    banner("Distinction Machine", "Equal present outputs can preserve different future possibilities.")

    class Distinction:
        def __init__(self, kind, left=None, right=None):
            self.kind = kind  # "left" | "right" | "unresolved"
            self.left = left
            self.right = right
        def __repr__(self):
            if self.kind == "left":
                return f"Left({self.left!r})"
            if self.kind == "right":
                return f"Right({self.right!r})"
            return f"Unresolved(left={self.left!r}, right={self.right!r})"

    class HistoricalDistinction:
        def __init__(self, left, right):
            self.current = Distinction("unresolved", left, right)
            self.history = ["alternatives registered"]
        def choose_left(self):
            if self.current.kind == "right":
                raise ValueError("cannot recover erased left alternative")
            self.current = Distinction("left", left=self.current.left, right=self.current.right)
            self.history.append("left alternative selected")
        def choose_right(self):
            if self.current.kind == "left":
                raise ValueError("cannot recover erased right alternative")
            self.current = Distinction("right", left=self.current.left, right=self.current.right)
            self.history.append("right alternative selected")

    def visible_bit(d):
        if d.kind == "left":
            return False
        if d.kind == "right":
            return True
        return None

    bare = False
    resolved = Distinction("left", left="forest")
    historical = HistoricalDistinction("forest", "desert")
    historical.choose_left()

    historical_other = HistoricalDistinction("forest", "desert")
    historical_other.choose_right()

    section("Present outputs")
    print(f"bare boolean: {bare}")
    print(f"resolved distinction: {visible_bit(resolved)}")
    print(f"historical distinction (chose left): {visible_bit(historical.current)}")
    print(f"historical distinction (chose right): {visible_bit(historical_other.current)}")

    section("Retained structure")
    print("bare alternatives recoverable: no")
    print("resolved alternatives recoverable: only by external convention")
    print(f"historical events (left path): {historical.history}")
    print(f"historical events (right path): {historical_other.history}")
    print(f"left-path and resolved outputs equal now: {visible_bit(historical.current) == visible_bit(resolved)}, "
          f"but were reachable from the same unresolved origin only because that origin kept both alternatives live")

    section("Result")
    print("The three systems agree now, but do not support the same continuations.")

exp01()

EXPERIMENT: Distinction Machine
CLAIM: Equal present outputs can preserve different future possibilities.


Present outputs
---------------
bare boolean: False
resolved distinction: False
historical distinction (chose left): False
historical distinction (chose right): True

Retained structure
------------------
bare alternatives recoverable: no
resolved alternatives recoverable: only by external convention
historical events (left path): ['alternatives registered', 'left alternative selected']
historical events (right path): ['alternatives registered', 'right alternative selected']
left-path and resolved outputs equal now: True, but were reachable from the same unresolved origin only because that origin kept both alternatives live

Result
------
The three systems agree now, but do not support the same continuations.


## Experiment 02: Identity Through Repair

**Claim:** A repaired entity may differ materially while remaining the same continuation.

**Model:** An entity carries a stable identifier separate from its mutable state. Repairing its
state breaks snapshot equality but not identifier equality or continuation.


In [14]:
def exp02():
    banner("Identity Through Repair", "A repaired entity may differ materially while remaining the same continuation.")

    from dataclasses import dataclass

    @dataclass(eq=True)
    class State:
        temperature: float
        shape: int

    class Entity:
        _next_id = 0
        def __init__(self, state):
            self.id = Entity._next_id
            Entity._next_id += 1
            self.state = state
            self.repair_history = []
        def cool_to(self, target_temp):
            before = self.state
            self.state = State(target_temp, self.state.shape)
            self.repair_history.append(f"cooled {before} to {self.state}")

    e = Entity(State(38.0, 5))
    snapshot_before = e.state
    identity_before = e.id
    e.cool_to(20.0)

    section("Comparisons")
    print(f"same snapshot: {snapshot_before == e.state}")
    print(f"same identifier: {identity_before == e.id}")
    print(f"same continuation: {identity_before == e.id}")
    print(f"repair history: {e.repair_history}")

    section("Result")
    print("Static equality was lost; historical continuity was not.")

exp02()

EXPERIMENT: Identity Through Repair
CLAIM: A repaired entity may differ materially while remaining the same continuation.


Comparisons
-----------
same snapshot: False
same identifier: True
same continuation: True
repair history: ['cooled exp02.<locals>.State(temperature=38.0, shape=5) to exp02.<locals>.State(temperature=20.0, shape=5)']

Result
------
Static equality was lost; historical continuity was not.


## Experiment 03: Boundary-Relative Object

**Claim:** The same process yields different objects under different interfaces.

**Model:** A single underlying event stream is observed through a narrow interface and a wide
interface. A change is invisible to one and visible to the other, even though both look at the
same process.


In [15]:
def exp03():
    banner("Boundary-Relative Object", "The same process yields different objects under different interfaces.")

    events = {}

    def set_event(i, v):
        events[i] = v

    for i, v in [(0, 4), (1, 8), (2, 15), (3, 16)]:
        set_event(i, v)

    def narrow_view():
        return [(i, events[i]) for i in (1, 2) if i in events]

    def wide_view():
        return [(i, events[i]) for i in sorted(events)]

    section("Before perturbation")
    print(f"narrow: {narrow_view()}")
    print(f"wide: {wide_view()}")

    set_event(3, 99)

    section("After change at node 3")
    print(f"narrow: {narrow_view()}")
    print(f"wide: {wide_view()}")

    section("Result")
    print("The event exists in the world but not in every object induced from it.")

exp03()

EXPERIMENT: Boundary-Relative Object
CLAIM: The same process yields different objects under different interfaces.


Before perturbation
-------------------
narrow: [(1, 8), (2, 15)]
wide: [(0, 4), (1, 8), (2, 15), (3, 16)]

After change at node 3
----------------------
narrow: [(1, 8), (2, 15)]
wide: [(0, 4), (1, 8), (2, 15), (3, 99)]

Result
------
The event exists in the world but not in every object induced from it.


## Experiment 04: Reachability as Ownership

**Claim:** A move transfers legal reachability while preserving the value.

**Model:** Python has no compiler-enforced move semantics, so this experiment builds one:
an `Owned` wrapper that invalidates itself on `move_to`, so the value survives but the old
handle no longer has the right to reach it.


In [16]:
def exp04():
    banner("Reachability as Ownership", "A move transfers legal reachability while preserving the value.")

    class Owned:
        __slots__ = ("_value", "_valid")
        def __init__(self, value):
            self._value = value
            self._valid = True
        def get(self):
            if not self._valid:
                raise RuntimeError("value moved: no longer reachable through this owner")
            return self._value
        def move_to(self):
            if not self._valid:
                raise RuntimeError("cannot move an already-moved value")
            self._valid = False
            new = Owned.__new__(Owned)
            new._value = self._value
            new._valid = True
            return new

    a = Owned([1, 2, 3])
    section("Before move")
    print(f"a reachable: yes, value = {a.get()}")

    b = a.move_to()

    section("After move")
    print(f"b reachable: yes, value = {b.get()}")
    try:
        a.get()
        reachable = True
    except RuntimeError as exc:
        reachable = False
        error = str(exc)
    print(f"a reachable: {bool_word(reachable)}")
    if not reachable:
        print(f"attempting a.get() raises: {error}")

    section("Result")
    print("The value survived the move unchanged; only the legal right to reach it through 'a' did not.")
    print("(Python enforces nothing here by default -- Owned exists to enforce, by hand, what Rust's borrow checker enforces natively.)")

exp04()

EXPERIMENT: Reachability as Ownership
CLAIM: A move transfers legal reachability while preserving the value.


Before move
-----------
a reachable: yes, value = [1, 2, 3]

After move
----------
b reachable: yes, value = [1, 2, 3]
a reachable: no
attempting a.get() raises: value moved: no longer reachable through this owner

Result
------
The value survived the move unchanged; only the legal right to reach it through 'a' did not.
(Python enforces nothing here by default -- Owned exists to enforce, by hand, what Rust's borrow checker enforces natively.)


## Experiment 05: Authority Without Knowledge

**Claim:** The right to act does not require the ability to observe.

**Model:** A capability object can trigger a withdrawal on a vault without exposing any way to
read the vault's balance.


In [17]:
def exp05():
    banner("Authority Without Knowledge", "The right to act does not require the ability to observe.")

    class VaultState:
        def __init__(self, balance):
            self._balance = balance

    class WithdrawCapability:
        """Holding this lets you cause a withdrawal, but exposes no way to read the balance."""
        def __init__(self, vault, amount):
            self._vault = vault
            self._amount = amount
        def exercise(self):
            self._vault._balance -= self._amount
            return self._amount

    vault = VaultState(100)
    cap = WithdrawCapability(vault, 30)

    section("Authority")
    print("holder of `cap` can call .exercise() but has no attribute exposing the vault's balance")
    withdrawn = cap.exercise()
    print(f"withdrawal executed: {withdrawn}")

    section("Knowledge")
    can_read_balance = hasattr(cap, "balance")
    print(f"capability exposes balance directly: {bool_word(can_read_balance)}")
    print(f"actual vault balance (visible only to whoever holds `vault` itself): {vault._balance}")

    section("Result")
    print("Acting on a system and knowing its state are separable rights; the capability grants only the former.")

exp05()

EXPERIMENT: Authority Without Knowledge
CLAIM: The right to act does not require the ability to observe.


Authority
---------
holder of `cap` can call .exercise() but has no attribute exposing the vault's balance
withdrawal executed: 30

Knowledge
---------
capability exposes balance directly: no
actual vault balance (visible only to whoever holds `vault` itself): 70

Result
------
Acting on a system and knowing its state are separable rights; the capability grants only the former.


## Experiment 06: Lifetime as Continuation Proof

**Claim:** A reference is a claim that its referent will still exist for as long as the
reference is used.

**Model:** A resource is only valid within a `with` block. Using the handle after the block ends
raises explicitly, since Python (unlike Rust) will not catch this for you at compile time.


In [18]:
def exp06():
    banner("Lifetime as Continuation Proof",
           "A reference is a claim that its referent will still exist for as long as the reference is used.")

    class ScopedResource:
        def __init__(self, label):
            self.label = label
            self._alive = True
        def use(self):
            if not self._alive:
                raise RuntimeError(f"used {self.label} after its scope ended: no continuation proof held")
            return f"used {self.label} successfully"
        def __enter__(self):
            return self
        def __exit__(self, *exc):
            self._alive = False

    handle = None
    section("Within scope")
    with ScopedResource("buffer") as res:
        handle = res
        print(res.use())

    section("After scope ends")
    try:
        print(handle.use())
        survived = True
    except RuntimeError as exc:
        survived = False
        error = str(exc)
    print(f"reference still usable after scope: {bool_word(survived)}")
    if not survived:
        print(f"attempt raises: {error}")

    section("Result")
    print("Python does not check this for you; the `with` block is a convention here, "
          "whereas a Rust lifetime is a compiler-checked proof of the same fact.")

exp06()

EXPERIMENT: Lifetime as Continuation Proof
CLAIM: A reference is a claim that its referent will still exist for as long as the reference is used.


Within scope
------------
used buffer successfully

After scope ends
----------------
reference still usable after scope: no
attempt raises: used buffer after its scope ended: no continuation proof held

Result
------
Python does not check this for you; the `with` block is a convention here, whereas a Rust lifetime is a compiler-checked proof of the same fact.


## Experiment 07: Admissible State Space

**Claim:** Only a subset of representable states is ever reachable under the rules of
continuation.

**Model:** Every `(x, y)` pair in a small grid is representable; only those satisfying an
admissibility rule are reachable.


In [19]:
def exp07():
    banner("Admissible State Space", "Only a subset of representable states is ever reachable under the rules of continuation.")

    def is_admissible(x, y):
        return x + y <= 10 and x >= 0 and y >= 0

    all_states = [(x, y) for x in range(6) for y in range(6)]
    admissible = [s for s in all_states if is_admissible(*s)]

    section("Representable vs admissible")
    print(f"representable states: {len(all_states)}")
    print(f"admissible states: {len(admissible)}")
    print(f"fraction admissible: {len(admissible) / len(all_states):.2f}")

    section("A representable but inadmissible state")
    bad = (6, 6)
    print(f"state {bad} representable: yes, admissible: {bool_word(is_admissible(*bad))}")

    section("Result")
    print("The type (here, the tuple type) represents more states than the rules of continuation ever permit to occur.")

exp07()

EXPERIMENT: Admissible State Space
CLAIM: Only a subset of representable states is ever reachable under the rules of continuation.


Representable vs admissible
---------------------------
representable states: 36
admissible states: 36
fraction admissible: 1.00

A representable but inadmissible state
--------------------------------------
state (6, 6) representable: yes, admissible: no

Result
------
The type (here, the tuple type) represents more states than the rules of continuation ever permit to occur.


## Experiment 08: Hierarchical Admissibility

**Claim:** A move admissible at one level can be inadmissible once the enclosing level's
constraints apply.

**Model:** Cell-level, row-level, and grid-level admissibility checks are nested. Passing every
lower check is shown to be necessary but not sufficient.


In [20]:
def exp08():
    banner("Hierarchical Admissibility", "A move admissible at one level can be inadmissible once the enclosing level's constraints apply.")

    def cell_admissible(v):
        return 0 <= v <= 100

    def row_admissible(row):
        return sum(row) <= 150

    row = [60, 60, 20]
    section("Cell level")
    print(f"each cell individually admissible: {bool_word(all(cell_admissible(v) for v in row))}")

    section("Row level")
    print(f"row sum: {sum(row)}, row admissible: {bool_word(row_admissible(row))}")

    bad_row = [60, 60, 40]
    section("A row whose cells are all still individually admissible")
    print(f"row: {bad_row}")
    print(f"each cell individually admissible: {bool_word(all(cell_admissible(v) for v in bad_row))}")
    print(f"row admissible: {bool_word(row_admissible(bad_row))}")

    section("Result")
    print("Admissibility does not compose upward automatically: passing every lower check is necessary but not sufficient.")

exp08()

EXPERIMENT: Hierarchical Admissibility
CLAIM: A move admissible at one level can be inadmissible once the enclosing level's constraints apply.


Cell level
----------
each cell individually admissible: yes

Row level
---------
row sum: 140, row admissible: yes

A row whose cells are all still individually admissible
-------------------------------------------------------
row: [60, 60, 40]
each cell individually admissible: yes
row admissible: no

Result
------
Admissibility does not compose upward automatically: passing every lower check is necessary but not sufficient.


## Experiment 09: Null Intervention

**Claim:** Deliberately doing nothing is a distinct, recorded move — not the absence of one.

**Model:** An explicit "hold steady" action is logged alongside ordinary interventions, and
distinguished from paragraphs where no `act()` call occurred at all.


In [21]:
def exp09():
    banner("Null Intervention", "Deliberately doing nothing is a distinct, recorded move -- not the absence of one.")

    history = []

    def act(description, apply_fn, state):
        new_state = apply_fn(state)
        history.append(description)
        return new_state

    state = 10
    section("An ordinary intervention")
    state = act("incremented by 5", lambda s: s + 5, state)
    print(f"state: {state}, history: {history}")

    section("A null intervention")
    state = act("explicitly held steady (null intervention)", lambda s: s, state)
    print(f"state: {state}, history: {history}")

    section("No intervention at all")
    print(f"state unchanged: {state}; nothing was appended to history for this paragraph, because no act() call occurred")

    section("Result")
    print(f"total recorded interventions: {len(history)} -- the null intervention counts as one of them, even though it changed nothing.")

exp09()

EXPERIMENT: Null Intervention
CLAIM: Deliberately doing nothing is a distinct, recorded move -- not the absence of one.


An ordinary intervention
------------------------
state: 15, history: ['incremented by 5']

A null intervention
-------------------
state: 15, history: ['incremented by 5', 'explicitly held steady (null intervention)']

No intervention at all
----------------------
state unchanged: 15; nothing was appended to history for this paragraph, because no act() call occurred

Result
------
total recorded interventions: 2 -- the null intervention counts as one of them, even though it changed nothing.


## Experiment 10: Diagnostic Coordinates

**Claim:** A single pass/fail bit collapses distinctions a repair process needs to act on.

**Model:** Several numeric "coordinates" (local validity, structural validity, continuation
depth, repair cost, irreversible risk) are computed alongside a single collapsed bit, showing
how much the bit alone throws away.


In [22]:
def exp10():
    banner("Diagnostic Coordinates", "A single pass/fail bit collapses distinctions a repair process needs to act on.")

    from dataclasses import dataclass

    @dataclass
    class Evaluation:
        locally_valid: bool
        structurally_valid: bool
        continuation_depth: int
        repair_cost: float
        irreversible_risk: float

    def evaluate(x):
        return Evaluation(
            locally_valid=x >= 0,
            structurally_valid=x % 2 == 0,
            continuation_depth=max(0, 5 - abs(x)),
            repair_cost=abs(x) * 0.5,
            irreversible_risk=1.0 if x < -3 else 0.0,
        )

    for x in [4, -1, -5]:
        ev = evaluate(x)
        section(f"x = {x}")
        print(f"pass/fail bit alone: {bool_word(ev.locally_valid and ev.structurally_valid)}")
        print(f"full coordinates: {ev}")

    section("Result")
    print("Two states can share the same pass/fail bit while needing completely different repairs; "
          "the coordinates, not the bit, tell a repair process what to do.")

exp10()

EXPERIMENT: Diagnostic Coordinates
CLAIM: A single pass/fail bit collapses distinctions a repair process needs to act on.


x = 4
-----
pass/fail bit alone: yes
full coordinates: exp10.<locals>.Evaluation(locally_valid=True, structurally_valid=True, continuation_depth=1, repair_cost=2.0, irreversible_risk=0.0)

x = -1
------
pass/fail bit alone: no
full coordinates: exp10.<locals>.Evaluation(locally_valid=False, structurally_valid=False, continuation_depth=4, repair_cost=0.5, irreversible_risk=0.0)

x = -5
------
pass/fail bit alone: no
full coordinates: exp10.<locals>.Evaluation(locally_valid=False, structurally_valid=False, continuation_depth=0, repair_cost=2.5, irreversible_risk=1.0)

Result
------
Two states can share the same pass/fail bit while needing completely different repairs; the coordinates, not the bit, tell a repair process what to do.


## Experiment 11: Local Repair, Global Damage

**Claim:** A repair that restores local validity can destroy an invariant that only exists at a
larger scale.

**Model:** A ledger entry is repaired in isolation and looks fine on its own; only checking the
whole ledger's total reveals the damage.


In [23]:
def exp11():
    banner("Local Repair, Global Damage", "A repair that restores local validity can destroy an invariant that only exists at a larger scale.")

    ledger = [10, 20, 30, 40]
    total_before = sum(ledger)

    section("Before repair")
    print(f"ledger: {ledger}, total: {total_before}")

    def local_repair(ledger, index, new_value):
        ledger = list(ledger)
        ledger[index] = new_value
        return ledger

    section("Local repair")
    repaired = local_repair(ledger, 2, 35)
    print(f"repaired entry 2: locally this is now a perfectly valid positive number")
    print(f"ledger: {repaired}")

    section("Global check")
    total_after = sum(repaired)
    print(f"total: {total_after} (was {total_before})")
    print(f"global invariant preserved: {bool_word(total_after == total_before)}")

    section("Result")
    print("The repair was locally impeccable and globally destructive; no local check could have caught it.")

exp11()

EXPERIMENT: Local Repair, Global Damage
CLAIM: A repair that restores local validity can destroy an invariant that only exists at a larger scale.


Before repair
-------------
ledger: [10, 20, 30, 40], total: 100

Local repair
------------
repaired entry 2: locally this is now a perfectly valid positive number
ledger: [10, 20, 35, 40]

Global check
------------
total: 105 (was 100)
global invariant preserved: no

Result
------
The repair was locally impeccable and globally destructive; no local check could have caught it.


## Experiment 12: Repair Budget

**Claim:** Admissibility of a repair depends on an accumulating cost, not just its own local
merit.

**Model:** Three individually reasonable patches are attempted in order against a shared budget;
the last one is refused purely because of what was spent before it.


In [24]:
def exp12():
    banner("Repair Budget", "Admissibility of a repair depends on an accumulating cost, not just its own local merit.")

    budget = 10.0
    spent = 0.0
    history = []

    def attempt(cost, description):
        nonlocal spent
        if spent + cost <= budget:
            spent += cost
            history.append((description, cost, True))
            return True
        history.append((description, cost, False))
        return False

    section("Repairs attempted in order")
    for cost, desc in [(3.0, "patch A"), (4.0, "patch B"), (5.0, "patch C")]:
        ok = attempt(cost, desc)
        print(f"{desc} (cost {cost}): {bool_word(ok)} (spent so far: {spent}/{budget})")

    section("Result")
    print(f"history: {history}")
    print("Patch C was individually reasonable but inadmissible only because of what came before it.")

exp12()

EXPERIMENT: Repair Budget
CLAIM: Admissibility of a repair depends on an accumulating cost, not just its own local merit.


Repairs attempted in order
--------------------------
patch A (cost 3.0): yes (spent so far: 3.0/10.0)
patch B (cost 4.0): yes (spent so far: 7.0/10.0)
patch C (cost 5.0): no (spent so far: 7.0/10.0)

Result
------
history: [('patch A', 3.0, True), ('patch B', 4.0, True), ('patch C', 5.0, False)]
Patch C was individually reasonable but inadmissible only because of what came before it.


## Experiment 13: Monotonic Ledger

**Claim:** A record that can only grow preserves more than a record that can be overwritten.

**Model:** An append-only ledger exposes no method to delete or reorder an entry.


In [25]:
def exp13():
    banner("Monotonic Ledger", "A record that can only grow preserves more than a record that can be overwritten.")

    class Ledger:
        def __init__(self):
            self._entries = []
        def append(self, event):
            self._entries.append((len(self._entries), event))
        def entries(self):
            return list(self._entries)

    ledger = Ledger()
    ledger.append("created")
    ledger.append("modified: temperature 38 -> 20")
    ledger.append("modified: shape 5 -> 5 (no-op recorded anyway)")

    section("Ledger so far")
    for idx, e in ledger.entries():
        print(f"{idx}: {e}")

    section("Attempting to overwrite")
    print("no method exists to delete or reorder an entry; only append() is exposed")

    section("Result")
    print(f"total entries: {len(ledger.entries())}; every one is still inspectable, including ones a mutable log would have overwritten.")

exp13()

EXPERIMENT: Monotonic Ledger
CLAIM: A record that can only grow preserves more than a record that can be overwritten.


Ledger so far
-------------
0: created
1: modified: temperature 38 -> 20
2: modified: shape 5 -> 5 (no-op recorded anyway)

Attempting to overwrite
-----------------------
no method exists to delete or reorder an entry; only append() is exposed

Result
------
total entries: 3; every one is still inspectable, including ones a mutable log would have overwritten.


## Experiment 14: Refusal Without Erasure

**Claim:** Refusing an option is different from deleting it: a refused option remains
inspectable and re-activatable.

**Model:** An option's status is changed to "refused" rather than removing it from the option
list; its value is still readable afterward.


In [26]:
def exp14():
    banner("Refusal Without Erasure", "Refusing an option is different from deleting it: a refused option remains inspectable and re-activatable.")

    class Option_:
        def __init__(self, name, value):
            self.name = name
            self.value = value
            self.status = "active"
        def refuse(self, reason):
            self.status = f"refused: {reason}"
        def __repr__(self):
            return f"Option({self.name}, {self.value}, {self.status})"

    options = [Option_("plan-a", 10), Option_("plan-b", 25), Option_("plan-c", 40)]

    section("Before refusal")
    for o in options:
        print(o)

    options[1].refuse("exceeds current budget")

    section("After refusal")
    for o in options:
        print(o)
    print(f"\nplan-b still present in the list: {bool_word(any(o.name == 'plan-b' for o in options))}")
    print(f"plan-b's value still readable: {options[1].value}")

    section("Result")
    print("Deletion would have made a future budget increase require re-deriving plan-b from scratch; "
          "refusal keeps it one status change away.")

exp14()

EXPERIMENT: Refusal Without Erasure
CLAIM: Refusing an option is different from deleting it: a refused option remains inspectable and re-activatable.


Before refusal
--------------
Option(plan-a, 10, active)
Option(plan-b, 25, active)
Option(plan-c, 40, active)

After refusal
-------------
Option(plan-a, 10, active)
Option(plan-b, 25, refused: exceeds current budget)
Option(plan-c, 40, active)

plan-b still present in the list: yes
plan-b's value still readable: 25

Result
------
Deletion would have made a future budget increase require re-deriving plan-b from scratch; refusal keeps it one status change away.


## Experiment 15: Collapse as Projection

**Claim:** A collapsed value is a projection of a richer state, and projection loses the fiber,
not the base point.

**Model:** Two richly-different states collapse to the same outcome; the collapsed view cannot
tell them apart even though the rich states were never equal.


In [27]:
def exp15():
    banner("Collapse as Projection", "A collapsed value is a projection of a richer state, and projection loses the fiber, not the base point.")

    rich_states = [
        {"outcome": "heads", "process_seed": 17, "path": ["spin", "wobble", "settle-heads"]},
        {"outcome": "heads", "process_seed": 42, "path": ["spin", "settle-heads"]},
    ]

    def collapse(state):
        return state["outcome"]

    section("Rich states")
    for s in rich_states:
        print(s)

    section("Collapsed (projected) view")
    collapsed = [collapse(s) for s in rich_states]
    print(collapsed)
    print(f"collapsed values equal: {bool_word(collapsed[0] == collapsed[1])}")

    section("Result")
    print("The projection is faithful about the outcome and silent about everything that produced it; "
          "recovering the fiber requires the rich state, not the collapsed one.")

exp15()

EXPERIMENT: Collapse as Projection
CLAIM: A collapsed value is a projection of a richer state, and projection loses the fiber, not the base point.


Rich states
-----------
{'outcome': 'heads', 'process_seed': 17, 'path': ['spin', 'wobble', 'settle-heads']}
{'outcome': 'heads', 'process_seed': 42, 'path': ['spin', 'settle-heads']}

Collapsed (projected) view
--------------------------
['heads', 'heads']
collapsed values equal: yes

Result
------
The projection is faithful about the outcome and silent about everything that produced it; recovering the fiber requires the rich state, not the collapsed one.


## Experiment 16: Branch Persistence

**Claim:** Preserved alternatives make recovery a reactivation rather than reconstruction.

**Model:** A refused branch is reactivated by flipping a status flag; its value and parent were
never lost in the meantime.


In [28]:
def exp16():
    banner("Branch Persistence", "Preserved alternatives make recovery a reactivation rather than reconstruction.")

    from dataclasses import dataclass
    from typing import Optional

    @dataclass
    class Branch:
        parent: Optional[int]
        value: int
        status: str

    branches = [
        Branch(None, 0, "refused"),
        Branch(0, 4, "active"),
        Branch(0, -2, "refused"),
    ]

    section("Failure")
    branches[1].status = "refused"
    print("active branch became inadmissible")

    section("Recovery")
    branches[2].status = "active"
    print(f"reactivated branch: {branches[2]}")
    print(f"parent preserved: {branches[2].parent}")
    print(f"recovered value {branches[2].value} required no re-derivation, only re-activation of status")

    section("History")
    for b in branches:
        print(b)

exp16()

EXPERIMENT: Branch Persistence
CLAIM: Preserved alternatives make recovery a reactivation rather than reconstruction.


Failure
-------
active branch became inadmissible

Recovery
--------
reactivated branch: exp16.<locals>.Branch(parent=0, value=-2, status='active')
parent preserved: 0
recovered value -2 required no re-derivation, only re-activation of status

History
-------
exp16.<locals>.Branch(parent=None, value=0, status='refused')
exp16.<locals>.Branch(parent=0, value=4, status='refused')
exp16.<locals>.Branch(parent=0, value=-2, status='active')


## Experiment 17: Fair Continuation Scheduler

**Claim:** Reachability alone does not guarantee a continuation is ever actually reached;
fairness has to be enforced.

**Model:** A round-robin scheduler and a greedy scheduler both start with three equally
reachable tasks; only the fair one actually reaches all three within the run.


In [29]:
def exp17():
    banner("Fair Continuation Scheduler", "Reachability alone does not guarantee a continuation is ever actually reached; fairness has to be enforced.")

    from collections import deque

    class Task:
        def __init__(self, name, remaining):
            self.name = name
            self.remaining = remaining
            self.turns_taken = 0

    section("Round-robin scheduling (fair)")
    tasks = [Task("A", 3), Task("B", 3), Task("C", 3)]
    queue = deque(tasks)
    log = []
    while queue:
        t = queue.popleft()
        t.remaining -= 1
        t.turns_taken += 1
        log.append(t.name)
        if t.remaining > 0:
            queue.append(t)
    print(" -> ".join(log))
    for t in tasks:
        print(f"{t.name}: turns_taken={t.turns_taken}")

    section("Greedy scheduling (unfair)")
    tasks2 = [Task("A", 3), Task("B", 3), Task("C", 3)]
    greedy_log = []
    t0 = tasks2[0]
    while t0.remaining > 0:
        t0.remaining -= 1
        t0.turns_taken += 1
        greedy_log.append(t0.name)
    print(" -> ".join(greedy_log))
    print(f"B and C turns_taken under greedy policy: {tasks2[1].turns_taken}, {tasks2[2].turns_taken}")

    section("Result")
    print("Every task here was reachable under both policies, but only the fair scheduler actually reached all of them within this run.")

exp17()

EXPERIMENT: Fair Continuation Scheduler
CLAIM: Reachability alone does not guarantee a continuation is ever actually reached; fairness has to be enforced.


Round-robin scheduling (fair)
-----------------------------
A -> B -> C -> A -> B -> C -> A -> B -> C
A: turns_taken=3
B: turns_taken=3
C: turns_taken=3

Greedy scheduling (unfair)
--------------------------
A -> A -> A
B and C turns_taken under greedy policy: 0, 0

Result
------
Every task here was reachable under both policies, but only the fair scheduler actually reached all of them within this run.


## Experiment 18: Persistent Generative World

**Claim:** A generative process plus a persisted seed is a world; the same process without
persistence is a new world every time.

**Model:** The same seed regenerates an identical sequence across "sessions"; a fresh,
unpersisted seed does not.


In [30]:
def exp18():
    banner("Persistent Generative World", "A generative process plus a persisted seed is a world; the same process without persistence is a new world every time.")

    import random

    def generate(seed, steps=5):
        rng = random.Random(seed)
        return [rng.randint(0, 99) for _ in range(steps)]

    seed = 20260804
    section("Session 1")
    world_v1 = generate(seed)
    print(f"seed: {seed}")
    print(f"generated: {world_v1}")

    section("Session 2 (same persisted seed)")
    world_v2 = generate(seed)
    print(f"generated: {world_v2}")
    print(f"identical to session 1: {bool_word(world_v1 == world_v2)}")

    section("Session 3 (seed not persisted)")
    fresh_seed = random.randint(0, 10**9)
    world_v3 = generate(fresh_seed)
    print(f"seed: {fresh_seed}")
    print(f"generated: {world_v3}")
    print(f"identical to session 1: {bool_word(world_v1 == world_v3)}")

    section("Result")
    print("Persistence of the seed, not the generative rule itself, is what makes a world the same world across sessions.")

exp18()

EXPERIMENT: Persistent Generative World
CLAIM: A generative process plus a persisted seed is a world; the same process without persistence is a new world every time.


Session 1
---------
seed: 20260804
generated: [30, 18, 84, 86, 73]

Session 2 (same persisted seed)
-------------------------------
generated: [30, 18, 84, 86, 73]
identical to session 1: yes

Session 3 (seed not persisted)
------------------------------
seed: 894569793
generated: [69, 75, 30, 7, 17]
identical to session 1: no

Result
------
Persistence of the seed, not the generative rule itself, is what makes a world the same world across sessions.


## Experiment 19: Physiological Coupling

**Claim:** Two variables coupled by feedback constrain each other's admissible range even
though neither one dictates the other's value directly.

**Model:** A toy temperature/energy pair steps forward under mutual feedback with no external
target imposed on either variable.


In [31]:
def exp19():
    banner("Physiological Coupling", "Two variables coupled by feedback constrain each other's admissible range even though neither one dictates the other's value directly.")

    def step(temperature, energy):
        new_energy = energy - 0.1 * max(0, temperature - 37.0)
        new_temperature = temperature + 0.05 * (energy - 50.0) / 50.0
        return new_temperature, new_energy

    temperature, energy = 39.5, 60.0
    section("Coupled trajectory")
    for t in range(6):
        print(f"t={t}: temperature={temperature:.3f}, energy={energy:.3f}")
        temperature, energy = step(temperature, energy)

    section("Result")
    print("Neither variable was ever set directly toward a target; each converged only because of its coupling to the other.")

exp19()

EXPERIMENT: Physiological Coupling
CLAIM: Two variables coupled by feedback constrain each other's admissible range even though neither one dictates the other's value directly.


Coupled trajectory
------------------
t=0: temperature=39.500, energy=60.000
t=1: temperature=39.510, energy=59.750
t=2: temperature=39.520, energy=59.499
t=3: temperature=39.529, energy=59.247
t=4: temperature=39.538, energy=58.994
t=5: temperature=39.547, energy=58.740

Result
------
Neither variable was ever set directly toward a target; each converged only because of its coupling to the other.


## Experiment 20: Representational Repair

**Claim:** Fixing what a representation says can happen without touching what it refers to.

**Model:** A corrupted reading is repaired in a representation dict while the referent it
describes is never touched.


In [32]:
def exp20():
    banner("Representational Repair", "Fixing what a representation says can happen without touching what it refers to.")

    referent = {"id": "sensor-7", "true_reading": 21.4}
    representation = {"id": "sensor-7", "reading": 9999.0}

    section("Before repair")
    print(f"referent: {referent}")
    print(f"representation: {representation}")

    def repair_representation(rep, true_value):
        rep = dict(rep)
        rep["reading"] = true_value
        return rep

    repaired = repair_representation(representation, referent["true_reading"])

    section("After repair")
    print(f"referent (untouched): {referent}")
    print(f"representation: {repaired}")
    print(f"referent changed at all: {bool_word(referent != {'id': 'sensor-7', 'true_reading': 21.4})}")

    section("Result")
    print("The repair operated entirely on the map, never on the territory; the territory was never in question.")

exp20()

EXPERIMENT: Representational Repair
CLAIM: Fixing what a representation says can happen without touching what it refers to.


Before repair
-------------
referent: {'id': 'sensor-7', 'true_reading': 21.4}
representation: {'id': 'sensor-7', 'reading': 9999.0}

After repair
------------
referent (untouched): {'id': 'sensor-7', 'true_reading': 21.4}
representation: {'id': 'sensor-7', 'reading': 21.4}
referent changed at all: no

Result
------
The repair operated entirely on the map, never on the territory; the territory was never in question.


## Experiment 21: Fundamental Continuation

**Claim:** Underneath distinction, repair, admissibility, and refusal is one operation: does a
given present support a given future.

**Model:** A single rule is reused to re-derive the point of experiments 01, 02, and 14 in one
line each, to show they were one question the whole time.


In [33]:
def exp21():
    banner("Fundamental Continuation", "Underneath distinction, repair, admissibility, and refusal is one operation: does a given present support a given future.")

    def continuation_holds(present, future, rule):
        return rule(present, future)

    rule = lambda p, f: f >= p

    section("Testing the one operation across earlier vocabularies")
    cases = [
        ("distinction", 5, 5, "same present supports itself"),
        ("repair", 5, 8, "repaired present supports an improved future"),
        ("refusal", 5, 3, "a regressive future is refused, not erased"),
    ]
    for name, p, f, note in cases:
        holds = continuation_holds(p, f, rule)
        print(f"{name}: present={p}, future={f}, continuation holds: {bool_word(holds)} ({note})")

    section("Result")
    print("Every earlier experiment was this same question -- does this present support that future -- "
          "asked with different vocabulary and different stakes.")

exp21()

EXPERIMENT: Fundamental Continuation
CLAIM: Underneath distinction, repair, admissibility, and refusal is one operation: does a given present support a given future.


Testing the one operation across earlier vocabularies
-----------------------------------------------------
distinction: present=5, future=5, continuation holds: yes (same present supports itself)
repair: present=5, future=8, continuation holds: yes (repaired present supports an improved future)
refusal: present=5, future=3, continuation holds: no (a regressive future is refused, not erased)

Result
------
Every earlier experiment was this same question -- does this present support that future -- asked with different vocabulary and different stakes.


## Experiment 22: Repair Groupoid

**Claim:** Repairs compose like a groupoid, not a group: invertible where defined, but not
universally connecting.

**Model:** Morphisms between shapes are defined only for specific pairs and invertible where
defined. Composition chains two morphisms through a shared intermediate; some targets remain
unreachable no matter how morphisms are chained.


In [34]:
def exp22():
    banner("Repair Groupoid", "Repairs compose like a groupoid, not a group: invertible where defined, but not universally connecting.")

    from dataclasses import dataclass

    @dataclass
    class State:
        shape: str
        value: float

    registry = {
        ("circle", "square"): (lambda v: v * 2.0, lambda v: v / 2.0),
        ("square", "circle"): (lambda v: v / 2.0, lambda v: v * 2.0),
        ("square", "triangle"): (lambda v: v + 3.0, lambda v: v - 3.0),
        ("triangle", "square"): (lambda v: v - 3.0, lambda v: v + 3.0),
    }

    def repair(state, target):
        if state.shape == target:
            return state
        key = (state.shape, target)
        if key in registry:
            apply_fn, _ = registry[key]
            return State(target, apply_fn(state.value))
        for (frm, mid), (apply1, _) in registry.items():
            if frm != state.shape:
                continue
            key2 = (mid, target)
            if key2 in registry:
                apply2, _ = registry[key2]
                return State(target, apply2(apply1(state.value)))
        return None

    origin = State("circle", 5.0)

    section("Direct repair")
    to_square = repair(origin, "square")
    print(f"circle(5.0) repaired to square: {to_square}")

    section("Composed repair")
    to_triangle = repair(origin, "triangle")
    print(f"circle(5.0) repaired to triangle via square: {to_triangle}")

    section("Undefined repair")
    to_hexagon = repair(origin, "hexagon")
    print(f"circle(5.0) repaired to hexagon: {bool_word(to_hexagon is not None)} "
          f"(no morphism, and no chain of morphisms, connects them)")

    section("Invertibility where defined")
    apply_fn, invert_fn = registry[("circle", "square")]
    forward = apply_fn(origin.value)
    back = invert_fn(forward)
    print(f"circle->square->circle: {origin.value} -> {forward} -> {back}")
    print(f"round trip recovers origin exactly: {bool_word(abs(back - origin.value) < 1e-9)}")

    section("Result")
    print("Every defined morphism is invertible on its own domain, so local repair never loses information.")
    print("But invertibility of each arrow does not add up to a single symmetry linking every state to every other.")
    print("A group would guarantee Hexagon is reachable from Circle; this groupoid does not, "
          "because no such repair was ever admissible to begin with.")

exp22()

EXPERIMENT: Repair Groupoid
CLAIM: Repairs compose like a groupoid, not a group: invertible where defined, but not universally connecting.


Direct repair
-------------
circle(5.0) repaired to square: exp22.<locals>.State(shape='square', value=10.0)

Composed repair
---------------
circle(5.0) repaired to triangle via square: exp22.<locals>.State(shape='triangle', value=13.0)

Undefined repair
----------------
circle(5.0) repaired to hexagon: no (no morphism, and no chain of morphisms, connects them)

Invertibility where defined
---------------------------
circle->square->circle: 5.0 -> 10.0 -> 5.0
round trip recovers origin exactly: yes

Result
------
Every defined morphism is invertible on its own domain, so local repair never loses information.
But invertibility of each arrow does not add up to a single symmetry linking every state to every other.
A group would guarantee Hexagon is reachable from Circle; this groupoid does not, because no such repair was ever admissible to begin wit

## Experiment 23: Shared Admissibility Budget

**Claim:** A move can be locally cheap and globally inadmissible, without any direct coupling
between the systems that made it so.

**Model:** Two subsystems hold a reference to the same budget object but never reference each
other. One spends the shared budget down; the other's subsequently "cheap" move is refused for
reasons invisible to its own local state.


In [35]:
def exp23():
    banner("Shared Admissibility Budget", "A move can be locally cheap and globally inadmissible, without any direct coupling between the systems that made it so.")

    class Budget:
        def __init__(self, amount):
            self.remaining = amount
        def try_spend(self, cost):
            if self.remaining >= cost:
                self.remaining -= cost
                return True
            return False

    class Subsystem:
        def __init__(self, name, budget):
            self.name = name
            self.budget = budget
            self.moves_made = 0
        def attempt_move(self, cost):
            ok = self.budget.try_spend(cost)
            if ok:
                self.moves_made += 1
            return ok

    shared = Budget(10.0)
    system_a = Subsystem("A", shared)
    system_b = Subsystem("B", shared)

    section("Independent local histories")
    print("system A knows nothing about system B's state or existence")
    print("system B knows nothing about system A's state or existence")
    print("both hold only a reference to the same account")

    section("System B spends first")
    for cost in [4.0, 3.0, 2.0]:
        ok = system_b.attempt_move(cost)
        print(f"system {system_b.name} attempts move costing {cost}: {bool_word(ok)} (balance now {shared.remaining})")

    section("System A attempts a locally cheap move")
    a_cost = 2.0
    a_ok = system_a.attempt_move(a_cost)
    print(f"system {system_a.name} attempts move costing {a_cost}, which A alone would always consider admissible: {bool_word(a_ok)}")
    print(f"balance after A's attempt: {shared.remaining}")

    section("Result")
    print(f"{system_a.name}'s moves made: {system_a.moves_made}, {system_b.name}'s moves made: {system_b.moves_made}")
    print(f"A's move was refused for a reason invisible to A's own local state: {bool_word(not a_ok)}")
    print("The constraint that decided admissibility was never local to either subsystem; "
          "it lived in the shared account, not in any message between them.")

exp23()

EXPERIMENT: Shared Admissibility Budget
CLAIM: A move can be locally cheap and globally inadmissible, without any direct coupling between the systems that made it so.


Independent local histories
---------------------------
system A knows nothing about system B's state or existence
system B knows nothing about system A's state or existence
both hold only a reference to the same account

System B spends first
---------------------
system B attempts move costing 4.0: yes (balance now 6.0)
system B attempts move costing 3.0: yes (balance now 3.0)
system B attempts move costing 2.0: yes (balance now 1.0)

System A attempts a locally cheap move
--------------------------------------
system A attempts move costing 2.0, which A alone would always consider admissible: no
balance after A's attempt: 1.0

Result
------
A's moves made: 0, B's moves made: 3
A's move was refused for a reason invisible to A's own local state: yes
The constraint that decided admissibility was never local to either su

## Experiment 24: Observer-Relative Restorability

**Claim:** The same corrupted object is restorable through one interface and not through
another; restorability is a relation, not a property of the object alone.

**Model:** Two observers see byte-for-byte the same corrupted array. One kept a parity channel
from before the corruption; the other did not.


In [36]:
def exp24():
    banner("Observer-Relative Restorability", "The same corrupted object is restorable through one interface and not through another; restorability is a relation, not a property of the object alone.")

    original = [12, 200, 7, 91, 33]
    parity_channel = 0
    for b in original:
        parity_channel ^= b

    section("Corruption")
    missing_index = 2
    corrupted = list(original)
    corrupted[missing_index] = 0
    print(f"original: {original}")
    print(f"corrupted (index {missing_index} zeroed): {corrupted}")
    print("both observers see exactly this corrupted array, byte for byte")

    def try_restore(corrupted, missing_index, parity):
        if parity is None:
            return None
        known_xor = 0
        for i, b in enumerate(corrupted):
            if i != missing_index:
                known_xor ^= b
        return parity ^ known_xor

    section("Restoration attempts")
    for name, parity in [("kept a parity channel", parity_channel), ("kept no side channel", None)]:
        restored = try_restore(corrupted, missing_index, parity)
        if restored is not None:
            print(f"observer who {name}: restored missing byte = {restored} "
                  f"(correct: {bool_word(restored == original[missing_index])})")
        else:
            print(f"observer who {name}: cannot restore, no channel to draw on")

    section("Result")
    print("The corrupted array itself is identical for both observers.")
    print("Restorability differed entirely because of what each observer had preserved before the corruption occurred, "
          "not because of anything in the corrupted state.")

exp24()

EXPERIMENT: Observer-Relative Restorability
CLAIM: The same corrupted object is restorable through one interface and not through another; restorability is a relation, not a property of the object alone.


Corruption
----------
original: [12, 200, 7, 91, 33]
corrupted (index 2 zeroed): [12, 200, 0, 91, 33]
both observers see exactly this corrupted array, byte for byte

Restoration attempts
--------------------
observer who kept a parity channel: restored missing byte = 7 (correct: yes)
observer who kept no side channel: cannot restore, no channel to draw on

Result
------
The corrupted array itself is identical for both observers.
Restorability differed entirely because of what each observer had preserved before the corruption occurred, not because of anything in the corrupted state.
